# Multiclass Advanced Curation & Control Class Curation (v5, v6, v7 Data)

This notebook implements advanced curation pipelines to clean the control/normal class and construct the final model training datasets for RoBERTa Multiclass versions v5, v6, and v7:

1. **Emotion Curation (v5 Data)**: Curates the control class with validated emotional posts while restoring standard mental health posts.
2. **Optimistic Control Curation (v6 Data)**: Curation including optimistic texts in the control class.
3. **Refined Suicidewatch Filtering (v7 Data - CAST Dataset)**: Standardizes clinical classes and applies noise filtering to the suicidewatch class.

---

## Setup & Imports
Load core packages and source datasets (`multiclass_df_augmented.csv` and `final_suicidewatch_filtered_dataset.csv`).

In [1]:
import pandas as pd
import re
from tqdm import tqdm

# Load source data
data_1 = pd.read_csv("multiclass_df_augmented.csv")
data_2 = pd.read_csv("final_suicidewatch_filtered_dataset.csv")

C:\Users\hana\AppData\Local\Temp\ipykernel_40556\2677691306.py:7: DtypeWarning: Columns (13,14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  data_2 = pd.read_csv("final_suicidewatch_filtered_dataset.csv")


## Section 1: Control Class Emotion & Leakage Inspection
Define and run a rule-based matching engine to filter out clinical leakage from the control class and assign emotional intensity scores.

In [2]:
import re
import pandas as pd
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

TEXT_COLUMN = "other_posts"

# Use these thresholds to review, not delete
HIGH_EMOTION_THRESHOLD = 5
MEDIUM_EMOTION_THRESHOLD = 2

# =========================================================
# NORMALIZATION
# =========================================================

def normalize_text(text):
    text = str(text).lower()

    # remove urls
    text = re.sub(r"http\S+|www\S+", " ", text)

    # remove usernames
    text = re.sub(r"@\w+", " ", text)

    # expand common informal contractions / slang
    replacements = {
        "im ": "i'm ",
        "i’m ": "i'm ",
        "dont ": "don't ",
        "cant ": "can't ",
        "wont ": "won't ",
        "ive ": "i've ",
        "idk": "i don't know",
        "tbh": "to be honest",
        "rn": "right now",
        "ngl": "not gonna lie",
        "af": "very",
        "kinda": "kind of",
        "sorta": "sort of",
        "wanna": "want to",
        "gonna": "going to",
        "gotta": "got to",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    # normalize repeated chars
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# =========================================================
# EMOTIONAL CONTROL PATTERNS
# =========================================================

CONTROL_EMOTION_PATTERNS = {

    # -----------------------------------------------------
    # BASIC POSITIVE EMOTION
    # -----------------------------------------------------
    "positive_emotion": {
        "score": 2,
        "patterns": [
            r"\bi'm happy\b",
            r"\bi am happy\b",
            r"\bi feel happy\b",
            r"\bi'm glad\b",
            r"\bi am glad\b",
            r"\bi feel good\b",
            r"\bi feel great\b",
            r"\bi'm excited\b",
            r"\bi am excited\b",
            r"\bi'm proud\b",
            r"\bi am proud\b",
            r"\bi feel proud\b",
            r"\bi'm grateful\b",
            r"\bi am grateful\b",
            r"\bi feel grateful\b",
            r"\bi'm thankful\b",
            r"\bi feel blessed\b",
            r"\bi'm relieved\b",
            r"\bi feel relieved\b",
            r"\bit made me smile\b",
            r"\bi smiled\b",
            r"\bi laughed\b",
            r"\bi'm in a good mood\b",
            r"\bi had a good day\b",
            r"\btoday was good\b",
            r"\btoday was great\b",
            r"\bi'm feeling better\b",
            r"\bi feel better\b",
            r"\bthings are getting better\b",
        ]
    },

    # -----------------------------------------------------
    # BASIC NEGATIVE EMOTION
    # -----------------------------------------------------
    "negative_emotion": {
        "score": 2,
        "patterns": [
            r"\bi'm sad\b",
            r"\bi am sad\b",
            r"\bi feel sad\b",
            r"\bi felt sad\b",
            r"\bi'm upset\b",
            r"\bi am upset\b",
            r"\bi feel upset\b",
            r"\bi'm angry\b",
            r"\bi am angry\b",
            r"\bi feel angry\b",
            r"\bi'm mad\b",
            r"\bi am mad\b",
            r"\bi'm annoyed\b",
            r"\bi am annoyed\b",
            r"\bi feel annoyed\b",
            r"\bi'm frustrated\b",
            r"\bi am frustrated\b",
            r"\bi feel frustrated\b",
            r"\bi'm disappointed\b",
            r"\bi am disappointed\b",
            r"\bi feel disappointed\b",
            r"\bi'm hurt\b",
            r"\bi feel hurt\b",
            r"\bi felt hurt\b",
            r"\bi'm emotional\b",
            r"\bi feel emotional\b",
            r"\bi cried\b",
            r"\bi've been crying\b",
            r"\bi was crying\b",
            r"\bi broke down\b",
            r"\bi had a bad day\b",
            r"\btoday was bad\b",
            r"\btoday sucked\b",
            r"\brough day\b",
            r"\bbad day at work\b",
            r"\bbad week\b",
            r"\brough week\b",
        ]
    },

    # -----------------------------------------------------
    # STRESS / OVERWHELM
    # -----------------------------------------------------
    "stress_overwhelm": {
        "score": 2,
        "patterns": [
            r"\bi'm stressed\b",
            r"\bi am stressed\b",
            r"\bi feel stressed\b",
            r"\bi've been stressed\b",
            r"\bi'm so stressed\b",
            r"\bstressed out\b",
            r"\bi'm overwhelmed\b",
            r"\bi am overwhelmed\b",
            r"\bi feel overwhelmed\b",
            r"\bi've been overwhelmed\b",
            r"\btoo much going on\b",
            r"\ba lot going on\b",
            r"\bi have a lot on my plate\b",
            r"\bmy plate is full\b",
            r"\bi can't keep up\b",
            r"\bi cant keep up\b",
            r"\bi'm exhausted\b",
            r"\bi am exhausted\b",
            r"\bi feel exhausted\b",
            r"\bi'm drained\b",
            r"\bi feel drained\b",
            r"\bi'm burned out\b",
            r"\bi am burned out\b",
            r"\bburnt out\b",
            r"\bi feel burnt out\b",
            r"\bi need a break\b",
            r"\bi need rest\b",
            r"\bi'm tired\b",
            r"\bi am tired\b",
            r"\bi'm so tired\b",
            r"\bi feel tired\b",
        ]
    },

    # -----------------------------------------------------
    # WORK / CAREER PRESSURE
    # -----------------------------------------------------
    "work_pressure": {
        "score": 2,
        "patterns": [
            r"\bwork is stressing me\b",
            r"\bwork has been stressful\b",
            r"\bwork is too much\b",
            r"\bwork is overwhelming\b",
            r"\bwork is loading me\b",
            r"\bworkload is too much\b",
            r"\bheavy workload\b",
            r"\bi'm overloaded at work\b",
            r"\bi am overloaded at work\b",
            r"\bi have too much work\b",
            r"\bmy job is stressful\b",
            r"\bmy job is draining\b",
            r"\bmy job is exhausting\b",
            r"\bi hate my job\b",
            r"\bi'm tired of my job\b",
            r"\bwork pressure\b",
            r"\bdeadline pressure\b",
            r"\bdeadlines are killing me\b",
            r"\bi missed a deadline\b",
            r"\bmy boss is stressing me\b",
            r"\btoxic workplace\b",
            r"\bbad day at the office\b",
            r"\blong day at work\b",
            r"\bovertime\b",
            r"\bworking late\b",
            r"\btoo many meetings\b",
            r"\bi feel underpaid\b",
            r"\bi feel undervalued at work\b",
            r"\bi'm not doing well at work\b",
            r"\bi am not doing well at work\b",
        ]
    },

    # -----------------------------------------------------
    # FAMILY / RELATIONSHIP RESPONSIBILITY
    # -----------------------------------------------------
    "family_relationship_pressure": {
        "score": 2,
        "patterns": [
            r"\bi'm not giving enough time to my family\b",
            r"\bi am not giving enough time to my family\b",
            r"\bi don't spend enough time with my family\b",
            r"\bi dont spend enough time with my family\b",
            r"\bi feel guilty about my family\b",
            r"\bi feel bad for my family\b",
            r"\bi'm neglecting my family\b",
            r"\bi am neglecting my family\b",
            r"\bfamily pressure\b",
            r"\bfamily problems\b",
            r"\bfamily issues\b",
            r"\bproblems at home\b",
            r"\bthings are hard at home\b",
            r"\bargued with my family\b",
            r"\bfight with my parents\b",
            r"\bfighting with my parents\b",
            r"\bfight with my mom\b",
            r"\bfight with my dad\b",
            r"\bmy parents are upset\b",
            r"\bmy parents don't understand\b",
            r"\brelationship problems\b",
            r"\brelationship issues\b",
            r"\bmy relationship is stressful\b",
            r"\bmy partner is upset\b",
            r"\bargued with my partner\b",
            r"\bfight with my partner\b",
            r"\bgoing through a breakup\b",
            r"\bwe broke up\b",
            r"\bbreakup\b",
            r"\bheartbroken\b",
            r"\bi miss them\b",
            r"\bi feel distant from them\b",
        ]
    },

    # -----------------------------------------------------
    # GUILT / REGRET / RESPONSIBILITY
    # -----------------------------------------------------
    "guilt_regret": {
        "score": 2,
        "patterns": [
            r"\bi feel guilty\b",
            r"\bi felt guilty\b",
            r"\bi'm guilty\b",
            r"\bi regret\b",
            r"\bi regretted\b",
            r"\bi feel bad about\b",
            r"\bi feel terrible about\b",
            r"\bi made a mistake\b",
            r"\bi messed up\b",
            r"\bi screwed up\b",
            r"\bi should have done better\b",
            r"\bi could have done better\b",
            r"\bi let them down\b",
            r"\bi disappointed them\b",
            r"\bi blame myself\b",
            r"\bit was my fault\b",
            r"\bi feel responsible\b",
            r"\bi'm not doing enough\b",
            r"\bi am not doing enough\b",
            r"\bi'm failing them\b",
            r"\bi am failing them\b",
        ]
    },

    # -----------------------------------------------------
    # ANXIETY / WORRY
    # -----------------------------------------------------
    "anxiety_worry": {
        "score": 2,
        "patterns": [
            r"\bi'm anxious\b",
            r"\bi am anxious\b",
            r"\bi feel anxious\b",
            r"\bi've been anxious\b",
            r"\bmy anxiety\b",
            r"\banxiety has been bad\b",
            r"\bi'm worried\b",
            r"\bi am worried\b",
            r"\bi feel worried\b",
            r"\bi worry about\b",
            r"\bi keep worrying\b",
            r"\bi can't stop worrying\b",
            r"\bi cant stop worrying\b",
            r"\bi'm nervous\b",
            r"\bi am nervous\b",
            r"\bi feel nervous\b",
            r"\bi'm scared\b",
            r"\bi am scared\b",
            r"\bi feel scared\b",
            r"\bi'm afraid\b",
            r"\bi am afraid\b",
            r"\bi feel afraid\b",
            r"\bpanic attack\b",
            r"\bi panicked\b",
            r"\bi'm panicking\b",
            r"\bi am panicking\b",
            r"\bi feel uneasy\b",
            r"\bi feel tense\b",
        ]
    },

    # -----------------------------------------------------
    # LONELINESS / SOCIAL EMOTION
    # -----------------------------------------------------
    "loneliness_social": {
        "score": 2,
        "patterns": [
            r"\bi'm lonely\b",
            r"\bi am lonely\b",
            r"\bi feel lonely\b",
            r"\bi felt lonely\b",
            r"\bi feel alone\b",
            r"\bi felt alone\b",
            r"\bi'm alone\b",
            r"\bi am alone\b",
            r"\bi miss my friends\b",
            r"\bi don't have friends\b",
            r"\bi dont have friends\b",
            r"\bi have no friends\b",
            r"\bi feel left out\b",
            r"\bi felt left out\b",
            r"\bi feel ignored\b",
            r"\bi felt ignored\b",
            r"\bnobody talks to me\b",
            r"\bno one talks to me\b",
            r"\bi feel invisible\b",
            r"\bi feel unwanted\b",
            r"\bi don't fit in\b",
            r"\bi dont fit in\b",
            r"\bi feel disconnected\b",
            r"\bi'm isolated\b",
            r"\bi am isolated\b",
        ]
    },

    # -----------------------------------------------------
    # SCHOOL / STUDY / EXAM STRESS
    # -----------------------------------------------------
    "school_stress": {
        "score": 2,
        "patterns": [
            r"\bschool is stressful\b",
            r"\bcollege is stressful\b",
            r"\buniversity is stressful\b",
            r"\bexam stress\b",
            r"\bexams are stressing me\b",
            r"\bfinals are stressing me\b",
            r"\bi failed my exam\b",
            r"\bi failed a test\b",
            r"\bi failed my class\b",
            r"\bi'm behind in school\b",
            r"\bi am behind in school\b",
            r"\bi'm behind on assignments\b",
            r"\bi am behind on assignments\b",
            r"\btoo much homework\b",
            r"\bassignment deadline\b",
            r"\bmy grades are bad\b",
            r"\bi'm worried about my grades\b",
            r"\bi am worried about my grades\b",
            r"\bstudy stress\b",
            r"\bi can't focus on studying\b",
            r"\bi cant focus on studying\b",
        ]
    },

    # -----------------------------------------------------
    # PERSONAL STRUGGLE / LIFE DIFFICULTY
    # -----------------------------------------------------
    "life_difficulty": {
        "score": 2,
        "patterns": [
            r"\blife has been hard\b",
            r"\blife is hard\b",
            r"\bthings have been hard\b",
            r"\bthings are hard\b",
            r"\bi'm going through a lot\b",
            r"\bi am going through a lot\b",
            r"\bi've been going through a lot\b",
            r"\bgoing through a rough time\b",
            r"\brough time lately\b",
            r"\bi'm struggling\b",
            r"\bi am struggling\b",
            r"\bi've been struggling\b",
            r"\bi struggle with\b",
            r"\bit's been difficult\b",
            r"\bit has been difficult\b",
            r"\bit's been rough\b",
            r"\bit has been rough\b",
            r"\bi'm having a hard time\b",
            r"\bi am having a hard time\b",
            r"\bi've had a hard time\b",
            r"\bi don't know what to do\b",
            r"\bi dont know what to do\b",
            r"\bi feel stuck\b",
            r"\bi'm stuck\b",
            r"\bi am stuck\b",
        ]
    },

    # -----------------------------------------------------
    # HEALTH / BODY / SLEEP
    # -----------------------------------------------------
    "health_sleep": {
        "score": 1,
        "patterns": [
            r"\bi'm sick\b",
            r"\bi am sick\b",
            r"\bi feel sick\b",
            r"\bi've been sick\b",
            r"\bi'm not feeling well\b",
            r"\bi am not feeling well\b",
            r"\bi feel awful physically\b",
            r"\bi feel terrible physically\b",
            r"\bi have insomnia\b",
            r"\bi can't sleep\b",
            r"\bi cant sleep\b",
            r"\bi couldn't sleep\b",
            r"\bi could not sleep\b",
            r"\bi barely slept\b",
            r"\bi haven't slept\b",
            r"\bi have not slept\b",
            r"\bi'm sleep deprived\b",
            r"\bi am sleep deprived\b",
            r"\bi'm exhausted from lack of sleep\b",
            r"\bi am exhausted from lack of sleep\b",
        ]
    },

    # -----------------------------------------------------
    # MONEY / FINANCIAL STRESS
    # -----------------------------------------------------
    "financial_stress": {
        "score": 2,
        "patterns": [
            r"\bmoney stress\b",
            r"\bfinancial stress\b",
            r"\bi'm broke\b",
            r"\bi am broke\b",
            r"\bi can't afford\b",
            r"\bi cant afford\b",
            r"\bi'm worried about money\b",
            r"\bi am worried about money\b",
            r"\bcan't pay rent\b",
            r"\bcant pay rent\b",
            r"\bcan't pay my bills\b",
            r"\bcant pay my bills\b",
            r"\bi'm in debt\b",
            r"\bi am in debt\b",
            r"\bdebt is stressing me\b",
            r"\bi lost my job\b",
            r"\bi got fired\b",
            r"\bi'm unemployed\b",
            r"\bi am unemployed\b",
            r"\bi need a job\b",
            r"\bjob hunting is stressful\b",
        ]
    },

    # -----------------------------------------------------
    # INFORMAL EMOTIONAL LANGUAGE
    # -----------------------------------------------------
    "informal_emotion": {
        "score": 1,
        "patterns": [
            r"\bfeeling down\b",
            r"\bfeeling low\b",
            r"\bi'm down bad\b",
            r"\bi am down bad\b",
            r"\bnot gonna lie i'm sad\b",
            r"\bnot gonna lie i feel bad\b",
            r"\btoday was trash\b",
            r"\btoday was ass\b",
            r"\bwork was trash\b",
            r"\bwork was ass\b",
            r"\blife is messy\b",
            r"\bi'm cooked\b",
            r"\bi am cooked\b",
            r"\bi'm so done with today\b",
            r"\bi am so done with today\b",
            r"\bi'm tired as hell\b",
            r"\bi am tired as hell\b",
            r"\bi'm stressed as hell\b",
            r"\bi am stressed as hell\b",
            r"\bthis week destroyed me\b",
            r"\bthis day destroyed me\b",
            r"\bi'm not okay today\b",
            r"\bi am not okay today\b",
            r"\bmentally tired\b",
            r"\bemotionally tired\b",
        ]
    }
}


# =========================================================
# OPTIONAL: RISK TERMS TO EXCLUDE FROM CONTROL-EMOTION REVIEW
# =========================================================
# This helps separate emotional-control samples from actual risk contamination.

RISK_LEAKAGE_PATTERNS = [
    r"\bkill myself\b",
    r"\bkms\b",
    r"\bend my life\b",
    r"\bwant to die\b",
    r"\bwanna die\b",
    r"\bsuicidal\b",
    r"\bsuicide\b",
    r"\bself harm\b",
    r"\bself-harm\b",
    r"\boverdose\b",
    r"\bslit my wrists\b",
    r"\bhanging myself\b",
    r"\bi don't want to live\b",
    r"\bi dont want to live\b",
    r"\bi wish i was dead\b",
    r"\bi wish i never woke up\b",
]


# =========================================================
# MATCHING
# =========================================================

def find_control_emotion_matches(text):
    matches = {}
    triggered_patterns = []
    total_score = 0
    total_hits = 0

    for category, info in CONTROL_EMOTION_PATTERNS.items():
        category_hits = []

        for pattern in info["patterns"]:
            found = re.findall(pattern, text)

            if found:
                category_hits.extend(found)
                total_hits += len(found)

                triggered_patterns.append({
                    "category": category,
                    "pattern": pattern,
                    "matches": found
                })

        if category_hits:
            matches[category] = list(set(category_hits))
            total_score += info["score"]

    # detect accidental suicide/self-harm contamination
    risk_hits = []

    for pattern in RISK_LEAKAGE_PATTERNS:
        found = re.findall(pattern, text)

        if found:
            risk_hits.extend(found)

    has_risk_leakage = len(risk_hits) > 0

    return {
        "emotion_matches": matches,
        "emotion_score": total_score,
        "emotion_hits": total_hits,
        "emotion_triggered_patterns": triggered_patterns,
        "has_risk_leakage": has_risk_leakage,
        "risk_hits": list(set(risk_hits))
    }


# =========================================================
# DECISION LOGIC
# =========================================================

def assign_control_emotion_decision(score, has_risk_leakage):
    if has_risk_leakage:
        return "REVIEW_RISK_LEAKAGE"

    if score >= HIGH_EMOTION_THRESHOLD:
        return "HIGH_EMOTION_CONTROL"

    if score >= MEDIUM_EMOTION_THRESHOLD:
        return "MEDIUM_EMOTION_CONTROL"

    return "LOW_OR_NEUTRAL_CONTROL"


# =========================================================
# MAIN PIPELINE
# =========================================================

def inspect_control_emotion_data(df, text_column=TEXT_COLUMN):
    tqdm.pandas()

    df = df.copy()

    df["clean_text"] = df[text_column].progress_apply(normalize_text)

    results = df["clean_text"].progress_apply(find_control_emotion_matches)

    df["emotion_matches"] = results.apply(lambda x: x["emotion_matches"])
    df["emotion_score"] = results.apply(lambda x: x["emotion_score"])
    df["emotion_hits"] = results.apply(lambda x: x["emotion_hits"])
    df["emotion_triggered_patterns"] = results.apply(lambda x: x["emotion_triggered_patterns"])
    df["has_risk_leakage"] = results.apply(lambda x: x["has_risk_leakage"])
    df["risk_hits"] = results.apply(lambda x: x["risk_hits"])

    df["word_count"] = df["clean_text"].apply(lambda x: len(x.split()))

    df["control_emotion_decision"] = df.apply(
        lambda row: assign_control_emotion_decision(
            row["emotion_score"],
            row["has_risk_leakage"]
        ),
        axis=1
    )

    return df


# =========================================================
# EXAMPLE USAGE
# =========================================================

control_df = data_1[data_1["multiclass_label"] == "control"].copy()

inspected_control = inspect_control_emotion_data(
    control_df,
    text_column="other_posts"
)

# inspected_control.to_csv("control_emotion_inspection.csv", index=False)

print(inspected_control["control_emotion_decision"].value_counts())

# high_emotion = inspected_control[
#     inspected_control["control_emotion_decision"] == "HIGH_EMOTION_CONTROL"
# ]

# medium_emotion = inspected_control[
#     inspected_control["control_emotion_decision"] == "MEDIUM_EMOTION_CONTROL"
# ]

# risk_leakage = inspected_control[
#     inspected_control["control_emotion_decision"] == "REVIEW_RISK_LEAKAGE"
# ]

# neutral_control = inspected_control[
#     inspected_control["control_emotion_decision"] == "LOW_OR_NEUTRAL_CONTROL"
# ]

# high_emotion[["text", "emotion_score", "emotion_matches"]].sample(20)

100%|██████████| 51803/51803 [10:38<00:00, 81.09it/s] 


control_emotion_decision
LOW_OR_NEUTRAL_CONTROL    44196
MEDIUM_EMOTION_CONTROL     7339
HIGH_EMOTION_CONTROL        239
REVIEW_RISK_LEAKAGE          29
Name: count, dtype: int64


### 1.2 Inspect Rule Results
Sample and save the inspected control class.

In [3]:
inspected_control.sample(100)

,author,subreddit,report_post,other_posts,is_self_report,is_control,self_report_sentence,selfdiag_score,selfdiag_matches,severe_flag,...,threeclass_label,clean_text,emotion_matches,emotion_score,emotion_hits,emotion_triggered_patterns,has_risk_leakage,risk_hits,word_count,control_emotion_decision
41920,ilovesalamence,relationships,NaN,Cheating ex (20f) already with someone else 1 ...,0,1,NaN,0.0,NaN,NaN,...,normal,cheating ex (20f) already with someone else 1 ...,{},0,0,[],False,[],50,LOW_OR_NEUTRAL_CONTROL
118155,anaislynn,relationships,NaN,I (20F) was scrolling through my boyfriend’s (...,0,1,NaN,0.0,NaN,NaN,...,normal,i (20f) was scrolling through my boyfriend’s (...,{},0,0,[],False,[],50,LOW_OR_NEUTRAL_CONTROL
126242,Thursdaythinker,relationships,NaN,How to help homesick country boy? Help! My boy...,0,1,NaN,0.0,NaN,NaN,...,normal,how to help homesick country boy? help! my boy...,{},0,0,[],False,[],50,LOW_OR_NEUTRAL_CONTROL
57070,Leavemealone098,relationships,NaN,Girlfriend's [21f] brothers physically attacke...,0,1,NaN,0.0,NaN,NaN,...,normal,girlfriend's [21f] brothers physically attacke...,{},0,0,[],False,[],194,LOW_OR_NEUTRAL_CONTROL
122451,DrELBrown,jokes,NaN,The good thing about meringues... ...is that t...,0,1,NaN,0.0,NaN,NaN,...,normal,the good thing about meringues.. ..is that the...,{},0,0,[],False,[],30,LOW_OR_NEUTRAL_CONTROL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88697,Askwomenthrowaway02,relationships,NaN,"Me [19 M] with my friend [18F] of 6ish months,...",0,1,NaN,0.0,NaN,NaN,...,normal,"me [19 m] with my friend [18f] of 6ish months,...",{},0,0,[],False,[],151,LOW_OR_NEUTRAL_CONTROL
63584,dark_zoroark,relationships,Boyfriend yelled at me for the first time?? So...,calling out of work tomorrow for i have a very...,1,0,"He said things to me like, “Deal with it.” and...",0.0,NaN,NaN,...,at_risk,calling out of work tomorrow for i have a very...,{},0,0,[],False,[],150,LOW_OR_NEUTRAL_CONTROL
71515,VeestraKildrak,relationships,NaN,My (23f) husband (24m) is not working and it's...,0,1,NaN,0.0,NaN,NaN,...,normal,my (23f) husband (24m) is not working and it's...,{'negative_emotion': ['i'm disappointed']},2,1,"[{'category': 'negative_emotion', 'pattern': '...",False,[],340,MEDIUM_EMOTION_CONTROL
3759,throwaway3948234,relationships,NaN,"Me [21 M] with my girlfriend [22 F] 2 years, m...",1,0,NaN,2.0,diag,NaN,...,at_risk,"me [21 m] with my girlfriend [22 f] 2 years, m...",{},0,0,[],False,[],154,LOW_OR_NEUTRAL_CONTROL


In [4]:
inspected_control.to_csv("control_emotion_inspection.csv", index=False)

### 1.3 Segment Control Class by Emotion Level
We filter the control class into neutral, emotional, and items to review for risk leakage.

In [5]:
clean_neutral_control = inspected_control[inspected_control["control_emotion_decision"] == "LOW_OR_NEUTRAL_CONTROL"]
emotional_control = inspected_control[
    inspected_control["control_emotion_decision"].isin(["MEDIUM_EMOTION_CONTROL", "HIGH_EMOTION_CONTROL"]
)]
risk_leakage_control = inspected_control[inspected_control["control_emotion_decision"] == "REVIEW_RISK_LEAKAGE"]

print("Emotion Segments summary:")
print(inspected_control["control_emotion_decision"].value_counts())

Emotion Segments summary:
control_emotion_decision
LOW_OR_NEUTRAL_CONTROL    44196
MEDIUM_EMOTION_CONTROL     7339
HIGH_EMOTION_CONTROL        239
REVIEW_RISK_LEAKAGE          29
Name: count, dtype: int64


### 1.4 Load Cured Control Data

In [6]:
inspected_control = pd.read_csv("control_emotion_inspection.csv")
print(inspected_control["control_emotion_decision"].value_counts())
print("Sample risk leakage posts:\n", inspected_control[inspected_control["control_emotion_decision"] == "REVIEW_RISK_LEAKAGE"]["other_posts"].head())

control_emotion_decision
LOW_OR_NEUTRAL_CONTROL    44196
MEDIUM_EMOTION_CONTROL     7339
HIGH_EMOTION_CONTROL        239
REVIEW_RISK_LEAKAGE          29
Name: count, dtype: int64
Sample risk leakage posts:
 1333    My Moms mental breakdown Hope this is the righ...
2562    I [18M] don't know if I should take our relati...
4104    Girlfriend broke up because I wasn't ready for...
4326    My (33/F) dad (65/M) literally plans to freeze...
5453    intermarried couples, how do you teach your ch...
Name: other_posts, dtype: object


## Section 2: Dataset Iterations Creation

### 2.1 Load Generated Cured Controls

In [7]:
emotional_control_examples = pd.read_csv("emotional_control_examples.csv")
emotional_control_examples['source'] = 'generated'
emotional_control_examples['multiclass_label'] = 'control'

long_emotional_control_examples = pd.read_csv("long_emotional_control_examples.csv")
long_emotional_control_examples['source'] = 'generated'
long_emotional_control_examples['multiclass_label'] = 'control'

mental_examples = data_1[data_1["multiclass_label"] != "control"]
accepted_control = inspected_control[inspected_control["emotion_score"] >= 1]

### 2.2 Create Version 5 Dataset (`generated_augmented_nonsuicidewatch_filtered_data.csv`)
Combines accepted control posts, generated emotional controls, and clinical examples (sampling 70%).

In [8]:
generated_augmented_nonsuicidewatch_filtered_data = pd.concat([
    accepted_control,
    emotional_control_examples,
    long_emotional_control_examples,
    mental_examples
], ignore_index=True)

generated_augmented_nonsuicidewatch_filtered_data = generated_augmented_nonsuicidewatch_filtered_data.sample(random_state=42, frac=0.7).reset_index(drop=True)
generated_augmented_nonsuicidewatch_filtered_data.to_csv("generated_augmented_nonsuicidewatch_filtered_data.csv", index=False)
print("Version 5 Dataset Saved! Shape:", generated_augmented_nonsuicidewatch_filtered_data.shape)

Version 5 Dataset Saved! Shape: (76343, 28)


### 2.3 Create Version 6 Dataset (`generated_augmented_nonsuicidewatch_filtered_optimisticadded_data.csv`)
Augment the version 5 dataset by incorporating optimistic control examples.

In [9]:
optimistic = pd.read_csv("optimistic_control_examples.csv")
optimistic['source'] = 'generated'
optimistic['multiclass_label'] = 'control'

generated_augmented_nonsuicidewatch_filtered_optimisticadded_data = pd.concat([
    accepted_control,
    emotional_control_examples,
    long_emotional_control_examples,
    mental_examples,
    optimistic
], ignore_index=True)

generated_augmented_nonsuicidewatch_filtered_optimisticadded_data = generated_augmented_nonsuicidewatch_filtered_optimisticadded_data.sample(random_state=42, frac=0.7).reset_index(drop=True)
generated_augmented_nonsuicidewatch_filtered_optimisticadded_data.to_csv("generated_augmented_nonsuicidewatch_filtered_optimisticadded_data.csv", index=False)
print("Version 6 Dataset Saved! Shape:", generated_augmented_nonsuicidewatch_filtered_optimisticadded_data.shape)

Version 6 Dataset Saved! Shape: (76763, 30)


### 2.4 Verify Class Distributions for v5 and v6

In [10]:
print("v5 Multiclass Distribution:\n", generated_augmented_nonsuicidewatch_filtered_data["multiclass_label"].value_counts())
print("v6 Multiclass Distribution:\n", generated_augmented_nonsuicidewatch_filtered_optimisticadded_data["multiclass_label"].value_counts())

v5 Multiclass Distribution:
 multiclass_label
suicidewatch     14663
adhd             12529
anxiety          10054
depression        9233
mentalhealth      8011
control           5478
socialanxiety     3000
bpd               2961
ptsd              2657
autism            1671
schizophrenia     1368
healthanxiety     1354
bipolarreddit     1018
edanonymous        989
addiction          541
lonely             443
alcoholism         373
Name: count, dtype: int64
v6 Multiclass Distribution:
 multiclass_label
suicidewatch     14671
adhd             12573
anxiety          10038
depression        9248
mentalhealth      7997
control           5902
socialanxiety     3035
bpd               2932
ptsd              2622
autism            1663
schizophrenia     1367
healthanxiety     1359
bipolarreddit     1008
edanonymous        996
addiction          546
lonely             438
alcoholism         368
Name: count, dtype: int64


### 2.5 Create Version 7 Dataset (`generated_augmented_suicidewatch_filtered_optimisticadded_data.csv`)
Replaces the clinical suicidewatch class with the noise-filtered version (`data_2`), keeps other clinical subreddits, adds optimistic/emotional controls, and appends a subset of non-emotional controls.

In [11]:
mental_examples_suicide_filtered = data_2[data_2['multiclass_label'] != "control"]
non_emotional_control = inspected_control[inspected_control["emotion_score"] < 2]
non_emotional_control_subset = non_emotional_control.iloc[:500]

generated_augmented_suicidewatch_filtered_optimisticadded_data = pd.concat([
    accepted_control,
    emotional_control_examples,
    long_emotional_control_examples,
    mental_examples_suicide_filtered,
    optimistic,
    non_emotional_control_subset
], ignore_index=True)

generated_augmented_suicidewatch_filtered_optimisticadded_data = generated_augmented_suicidewatch_filtered_optimisticadded_data.sample(random_state=42, frac=0.7).reset_index(drop=True)
generated_augmented_suicidewatch_filtered_optimisticadded_data.to_csv("generated_augmented_suicidewatch_filtered_optimisticadded_data.csv", index=False)
print("Version 7 Dataset Saved! Shape:", generated_augmented_suicidewatch_filtered_optimisticadded_data.shape)
print("v7 Multiclass Distribution:\n", generated_augmented_suicidewatch_filtered_optimisticadded_data["multiclass_label"].value_counts())

Version 7 Dataset Saved! Shape: (64791, 33)
v7 Multiclass Distribution:
 multiclass_label
adhd             12650
anxiety          10133
depression        9152
mentalhealth      7995
control           6245
socialanxiety     3017
bpd               2957
ptsd              2635
suicidewatch      2418
autism            1680
schizophrenia     1307
healthanxiety     1280
bipolarreddit     1022
edanonymous        979
addiction          517
lonely             436
alcoholism         368
Name: count, dtype: int64
